# Protein Prediction: Setup And Exploration

Use the `Protein Prediction (UdonPred uv)` kernel. This notebook is for quick setup checks, small UdonPred inference runs, and first TriZOD score exploration.

## 1. Check Environment

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "UdonPred").exists():
    ROOT = ROOT.parent

print("Python:", sys.version)
print("Project root:", ROOT)
print("UdonPred exists:", (ROOT / "UdonPred").exists())
print("TriZOD exists:", (ROOT / "TriZOD").exists())

In [ ]:
import importlib

modules = [
    "Bio",
    "datasets",
    "onnxruntime",
    "torch",
    "transformers",
    "pandas",
    "polars",
    "sklearn",
    "scipy",
    "yaml",
]

for name in modules:
    module = importlib.import_module(name)
    print(f"{name}: OK {getattr(module, '__version__', '')}")

## 2. Run A Small UdonPred Prediction

The first run may take several minutes because it downloads the ProstT5 backbone from Hugging Face.

In [ ]:
import subprocess

cmd = [
    sys.executable,
    "predict.py",
    str(ROOT / "examples" / "smoke.fasta"),
    "weights",
    "--target",
    "trizod",
    "--device",
    "cpu",
    "--batch-size",
    "200",
    "--smooth",
    "0",
]

result = subprocess.run(
    cmd,
    cwd=ROOT / "UdonPred",
    text=True,
    capture_output=True,
    timeout=900,
)

print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
result.check_returncode()

## 3. Inspect Available TriZOD Files

In [ ]:
trizod_dir = ROOT / "TriZOD"
for path in sorted(trizod_dir.glob("*")):
    print(path.name)

## 4. Load TriZOD JSONL Files

The TriZOD files use newline-delimited JSON records, despite the `.json` extension. This cell loads each tier and shows the record shape so the group can agree on the parsing code before deeper analysis.

In [ ]:
import json
import pandas as pd

def read_jsonl(path):
    with path.open() as handle:
        return [json.loads(line) for line in handle if line.strip()]

tiers = ["unfiltered", "tolerant", "moderate", "strict"]
records = []

for tier in tiers:
    path = trizod_dir / f"{tier}.json"
    if not path.exists():
        continue
    data = read_jsonl(path)
    first = data[0] if data else {}
    records.append({
        "tier": tier,
        "records": len(data),
        "columns": list(first.keys()) if isinstance(first, dict) else [],
        "first_id": first.get("ID") if isinstance(first, dict) else None,
        "first_sequence_length": len(first.get("seq", "")) if isinstance(first, dict) else None,
    })

pd.DataFrame(records)

## 5. Plot G-Score Distributions

Each TriZOD record has a `gscores` list with one value per residue, plus `None` where no score is available.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

rows = []
for tier in tiers:
    path = trizod_dir / f"{tier}.json"
    if not path.exists():
        continue
    for record in read_jsonl(path):
        for position, score in enumerate(record.get("gscores", []), start=1):
            if score is not None and 0 <= score <= 1:
                rows.append({
                    "tier": tier,
                    "id": record.get("ID"),
                    "entry_id": record.get("entryID"),
                    "position": position,
                    "g_score": float(score),
                })

gscore_df = pd.DataFrame(rows)
print(gscore_df.groupby("tier")["g_score"].describe() if not gscore_df.empty else "No G-scores found")

if not gscore_df.empty:
    sns.histplot(data=gscore_df, x="g_score", hue="tier", element="step", stat="density", common_norm=False)
    plt.xlim(0, 1)
    plt.title("TriZOD G-score distributions")
    plt.show()

## 6. Compare TriZOD Filtering Tiers

Compare unfiltered, tolerant, moderate, and strict TriZOD G-score distributions and overlap. This is useful for understanding how filtering changes the available data and score distribution.

In [ ]:
trizod_records = {}
for tier in tiers:
    path = trizod_dir / f"{tier}.json"
    trizod_records[tier] = read_jsonl(path) if path.exists() else []

tier_summary_rows = []
for tier, records_for_tier in trizod_records.items():
    scored_residues = 0
    total_residues = 0
    protein_ids = set()
    entry_ids = set()
    for record in records_for_tier:
        protein_ids.add(record.get("ID"))
        entry_ids.add(record.get("entryID"))
        total_residues += len(record.get("seq", ""))
        scored_residues += sum(score is not None for score in record.get("gscores", []))
    tier_summary_rows.append({
        "tier": tier,
        "records": len(records_for_tier),
        "unique_entry_ids": len(entry_ids),
        "unique_record_ids": len(protein_ids),
        "total_residues": total_residues,
        "scored_residues": scored_residues,
        "scored_fraction": scored_residues / total_residues if total_residues else 0,
    })

tier_summary_df = pd.DataFrame(tier_summary_rows)
display(tier_summary_df)

In [ ]:
gscore_by_tier = {}
for tier, records_for_tier in trizod_records.items():
    values = []
    for record in records_for_tier:
        values.extend(score for score in record.get("gscores", []) if score is not None)
    gscore_by_tier[tier] = pd.Series(values, name=tier, dtype="float64")

gscore_summary_df = pd.DataFrame({
    tier: series.describe()
    for tier, series in gscore_by_tier.items()
}).T
display(gscore_summary_df.style.format("{:.4f}"))

In [ ]:
plt.figure(figsize=(10, 5))
for tier, series in gscore_by_tier.items():
    sns.kdeplot(series, label=tier, clip=(0, 1), common_norm=False)
plt.xlim(0, 1)
plt.xlabel("G-score")
plt.ylabel("Density")
plt.title("TriZOD G-score distributions by filtering tier")
plt.legend(title="Tier")
plt.tight_layout()
plt.show()

In [ ]:
id_sets = {
    tier: {record.get("ID") for record in records_for_tier}
    for tier, records_for_tier in trizod_records.items()
}

overlap_rows = []
for left in tiers:
    for right in tiers:
        left_ids = id_sets.get(left, set())
        right_ids = id_sets.get(right, set())
        intersection = left_ids & right_ids
        union = left_ids | right_ids
        overlap_rows.append({
            "left": left,
            "right": right,
            "shared_records": len(intersection),
            "jaccard": len(intersection) / len(union) if union else 0,
        })

overlap_df = pd.DataFrame(overlap_rows)
overlap_matrix = overlap_df.pivot(index="left", columns="right", values="jaccard")

plt.figure(figsize=(5, 4))
sns.heatmap(overlap_matrix, annot=True, fmt=".2f", cmap="mako", vmin=0, vmax=1)
plt.title("TriZOD record overlap by filtering tier")
plt.tight_layout()
plt.show()

display(overlap_df.pivot(index="left", columns="right", values="shared_records"))

In [ ]:
def record_score_frame(tier, records_for_tier):
    rows = []
    for record in records_for_tier:
        record_id = record.get("ID")
        for position, score in enumerate(record.get("gscores", []), start=1):
            if score is not None:
                rows.append({"ID": record_id, "position": position, tier: float(score)})
    return pd.DataFrame(rows)

tier_score_frames = {
    tier: record_score_frame(tier, records_for_tier)
    for tier, records_for_tier in trizod_records.items()
}

comparison_pairs = [("unfiltered", "tolerant"), ("tolerant", "moderate"), ("moderate", "strict"), ("unfiltered", "strict")]
difference_rows = []

for left, right in comparison_pairs:
    left_df = tier_score_frames[left]
    right_df = tier_score_frames[right]
    merged = left_df.merge(right_df, on=["ID", "position"], how="inner")
    diff = merged[right] - merged[left]
    difference_rows.append({
        "comparison": f"{right} - {left}",
        "shared_scored_residues": len(merged),
        "mean_difference": diff.mean() if len(diff) else None,
        "median_abs_difference": diff.abs().median() if len(diff) else None,
        "max_abs_difference": diff.abs().max() if len(diff) else None,
        "correlation": merged[left].corr(merged[right], method="spearman") if len(merged) else None,
    })

tier_difference_df = pd.DataFrame(difference_rows)
display(tier_difference_df.style.format({
    "mean_difference": "{:.4f}",
    "median_abs_difference": "{:.4f}",
    "max_abs_difference": "{:.4f}",
    "correlation": "{:.4f}",
}))

## 7. Check UdonPred Evaluation Data

The 7x7 matrix needs `test.fasta` and `test.jsonl` for each UdonPred dataset.

In [ ]:
DATASETS = ["trizod", "chezod", "softdis", "pdbflex", "atlas", "plddt", "disprot"]

rows = []
for dataset in DATASETS:
    dataset_dir = ROOT / "UdonPred" / "data" / dataset
    test_fasta = dataset_dir / "test.fasta"
    test_jsonl = dataset_dir / "test.jsonl"
    rows.append({
        "dataset": dataset,
        "test_fasta": test_fasta.exists(),
        "test_jsonl": test_jsonl.exists(),
        "jsonl_records": sum(1 for _ in test_jsonl.open()) if test_jsonl.exists() else 0,
        "fasta_headers": sum(1 for line in test_fasta.open() if line.startswith(">")) if test_fasta.exists() else 0,
    })

data_check_df = pd.DataFrame(rows)
display(data_check_df)
assert data_check_df[["test_fasta", "test_jsonl"]].all().all(), "Missing one or more UdonPred test files."

## 8. Run The 7x7 Matrix

This launches 49 prediction jobs. On CPU, it can take many hours because ProstT5 embeddings are computed for every test set. Set `RUN_FULL_MATRIX = True` when you are ready to run it. Use `DEVICE = "cuda"` on a GPU machine.

In [ ]:
import subprocess

RUN_FULL_MATRIX = False
DEVICE = "cpu"  # change to "cuda" on a GPU machine

matrix_cmd = [
    sys.executable,
    str(ROOT / "scripts" / "run_udonpred_matrix.py"),
    "--device",
    DEVICE,
]

print(" ".join(matrix_cmd))

if RUN_FULL_MATRIX:
    subprocess.run(matrix_cmd, cwd=ROOT, check=True)
else:
    print("Skipped. Set RUN_FULL_MATRIX = True to start the full 7x7 run.")

If prediction files already exist and you only want to recompute metrics, run this cell.

In [ ]:
RECOMPUTE_METRICS_ONLY = False

metrics_cmd = [
    sys.executable,
    str(ROOT / "scripts" / "run_udonpred_matrix.py"),
    "--skip-predictions",
]

print(" ".join(metrics_cmd))

if RECOMPUTE_METRICS_ONLY:
    subprocess.run(metrics_cmd, cwd=ROOT, check=True)
else:
    print("Skipped. Set RECOMPUTE_METRICS_ONLY = True after predictions exist.")

## 9. Evaluate Matrix Results

The matrix uses Spearman correlation for continuous test sets. For DisProt, it reports average precision and AUROC.

In [ ]:
matrix_path = ROOT / "results" / "udonpred_matrix" / "matrix.csv"

if not matrix_path.exists():
    raise FileNotFoundError(
        f"Matrix file not found: {matrix_path}\n"
        "Run the full matrix cell first, or run `python scripts/run_udonpred_matrix.py --device cpu` from the repo root."
    )

matrix_df = pd.read_csv(matrix_path).set_index("train_dataset")
matrix_numeric = matrix_df.apply(pd.to_numeric)
display(matrix_numeric.style.format("{:.3f}"))

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(matrix_numeric, annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1)
plt.xlabel("Test dataset / metric")
plt.ylabel("Training dataset")
plt.title("UdonPred cross-dataset evaluation matrix")
plt.tight_layout()
plt.show()

In [ ]:
continuous_tests = ["trizod", "chezod", "softdis", "pdbflex", "atlas", "plddt"]

best_rows = []
for test_dataset in continuous_tests + ["disprot\n(AP)", "disprot\n(AUROC)"]:
    best_train = matrix_numeric[test_dataset].idxmax()
    best_score = matrix_numeric.loc[best_train, test_dataset]
    diagonal_score = matrix_numeric.loc[test_dataset, test_dataset] if test_dataset in matrix_numeric.index else None
    best_rows.append({
        "test_metric": test_dataset,
        "best_training_dataset": best_train,
        "best_score": best_score,
        "same_dataset_score": diagonal_score,
        "headroom_vs_same_dataset": best_score - diagonal_score if diagonal_score is not None else None,
    })

evaluation_summary_df = pd.DataFrame(best_rows)
display(evaluation_summary_df.style.format({
    "best_score": "{:.3f}",
    "same_dataset_score": "{:.3f}",
    "headroom_vs_same_dataset": "{:.3f}",
}))

In [ ]:
diagonal = pd.Series(
    {dataset: matrix_numeric.loc[dataset, dataset] for dataset in continuous_tests},
    name="same_dataset_spearman",
)

off_diagonal_values = []
for train_dataset in continuous_tests:
    for test_dataset in continuous_tests:
        if train_dataset != test_dataset:
            off_diagonal_values.append(matrix_numeric.loc[train_dataset, test_dataset])

print("Mean same-dataset Spearman:", round(diagonal.mean(), 3))
print("Mean cross-dataset Spearman:", round(pd.Series(off_diagonal_values).mean(), 3))
display(diagonal.to_frame())

## 10. Next Project Tasks

- Build null baselines with shuffled labels.
- Compare simple ensemble strategies across predictors.
- Use the matrix summary to decide which dataset definitions transfer well and which ones disagree.